In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations

In [ ]:
# Define folder
data_dir = Path("..") / "data" / "processed"

# Read the two files individually
v1 = pd.read_csv(data_dir / "dropout_version1_data.csv")
v2 = pd.read_csv(data_dir / "dropout_version2_data.csv")

# Quick check
print("v1 shape:", v1.shape)
print("v2 shape:", v2.shape)


In [ ]:
load_dotenv()

RAW_DIR = Path(os.getenv("STUDENT_RAW_DIR")).expanduser()
OUT_DIR = Path(os.getenv("OUTPUTS_DIR")).expanduser()

for p in (RAW_DIR, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Read the two files individually, dropping any unnamed index column
v1 = pd.read_csv(OUT_DIR / "dropout_version1_data.csv").loc[:, ~pd.read_csv(OUT_DIR / "dropout_version1_data.csv").columns.str.contains('^Unnamed')]
v2 = pd.read_csv(OUT_DIR / "dropout_version2_data.csv").loc[:, ~pd.read_csv(OUT_DIR / "dropout_version2_data.csv").columns.str.contains('^Unnamed')]

# Quick check
print("v1 shape:", v1.shape)
print("v2 shape:", v2.shape)


In [ ]:
v1

In [ ]:
v2

In [ ]:
# Add version-specific suffixes (_v1 and _v2) to columns in v1 and v2 (except SINH_ID),
# then merge the two datasets on SINH_ID to create a combined DataFrame.

# Add suffixes but keep SINH_ID unchanged
v1 = v1.rename(columns={col: f"{col}_v1" for col in v1.columns if col != "SINH_ID"})
v2 = v2.rename(columns={col: f"{col}_v2" for col in v2.columns if col != "SINH_ID"})

# Combine on SINH_ID
data = pd.merge(v1, v2, on="SINH_ID", how="inner")

print("combined shape:", data.shape)


In [ ]:
data

In [ ]:
# Find identical (100% match) and near-identical (>threshold match) column pairs in `data`.
# Treat NaNs in the same positions as equal; for numeric values, also allow small differences via np.isclose.

threshold = 0.70  # near-identical threshold (e.g., 0.95 = 95%)

identical_pairs = []
near_identical = []  # (col1, col2, match_rate)

cols = list(data.columns)

for c1, c2 in combinations(cols, 2):
    s1 = data[c1]
    s2 = data[c2]

    # Fast path: exact equality (NaNs at same positions count as equal)
    if s1.equals(s2):
        identical_pairs.append((c1, c2))
        continue

    # Build a tolerant equality mask:
    # 1) strict equality (works for strings, categoricals, exact numbers)
    eq_strict = s1.eq(s2)

    # 2) count NaNs at same positions as equal
    both_nan = s1.isna() & s2.isna()

    # 3) numeric closeness for entries that can be parsed as numbers
    s1_num = pd.to_numeric(s1, errors="coerce")
    s2_num = pd.to_numeric(s2, errors="coerce")
    num_valid = s1_num.notna() & s2_num.notna()
    close_num = pd.Series(False, index=data.index)
    if num_valid.any():
        close_num.loc[num_valid] = np.isclose(
            s1_num.loc[num_valid].to_numpy(),
            s2_num.loc[num_valid].to_numpy(),
            rtol=1e-5,
            atol=1e-8,
            equal_nan=True,
        )

    # Combine all equality notions
    eq = eq_strict | both_nan | close_num

    # Match rate across all rows
    match_rate = float(eq.mean())

    if match_rate >= threshold:
        near_identical.append((c1, c2, match_rate))

# Report
if identical_pairs:
    print("Identical column pairs (100% match):")
    for a, b in identical_pairs:
        print(f"  - {a} == {b}")

if near_identical:
    print(f"\nNear-identical column pairs (>= {int(threshold*100)}% match):")
    for a, b, r in sorted(near_identical, key=lambda x: -x[2]):
        print(f"  - {a} ~ {b}: {r:.2%} match")

if not identical_pairs and not near_identical:
    print("No identical or near-identical column pairs found.")


In [ ]:
# Delete (I will go for deleting colummns from version one, the old dataset) one of the identical columns

del data['Student_num_v1']
del data['Study_Pro_Start_Year_v1']
del data['Prior_Edu_Country_v1']
del data['Study_Prog_Name_v1']
del data['Study_Prog_Exam_Completed_v1']
data

In [ ]:
print(data['Exit_Status_v1'].value_counts(dropna=False))

In [ ]:
print(data['sdo5_drop_out_v2'].value_counts(dropna=False))

In [ ]:
print(data['sdo5_drop_out_with_degree_v2'].value_counts(dropna=False))

In [ ]:
print(data['sdo5_drop_out_to_other_degree_in_HU_v2'].value_counts(dropna=False))

# Continue cleaning combined data set

In [ ]:
# Show value counts for object-type columns with <= 50 unique categories

object_cols = data.select_dtypes(include=["object"]).columns

for col in object_cols:
    n_unique = data[col].nunique(dropna=True)
    if n_unique <= 100:
        print(f"\nColumn: {col}  (unique categories: {n_unique})")
        print(data[col].value_counts(dropna=False))


--------------------------------- Handle categories in Prior_Edu_School_Name_v1 column ------------------------------------------------

In [ ]:
pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
pd.set_option('display.expand_frame_repr', False)  

In [ ]:
print(data['Prior_Edu_School_Name_v1'].value_counts(dropna=False))

In [ ]:
# Clean categorical column (Prior_Edu_School_Name_v1):
# - Replace categories with fewer than 100 occurrences with 'Other'
# - Replace '0' (string or numeric) with 'Unknown'

col = "Prior_Edu_School_Name_v1"
threshold = 100

# Step 1: Replace rare categories with "Other"
value_counts = data[col].value_counts()
rare_cats = value_counts[value_counts < threshold].index
data[col] = data[col].replace(rare_cats, "Other")

# Step 2: Replace "0" (string or numeric) with "Unknown"
data[col] = data[col].replace([0, "0"], "Unknown")

# Quick check
print(data[col].value_counts(dropna=False))


In [ ]:
# Clean "Prior_Edu_School_Name_v1":
# - Normalize text (trim, unify variants)
# - Consolidate obvious duplicates via regex rules
# - Replace spaces with underscores for consistency

import re
import pandas as pd

col = "Prior_Edu_School_Name_v1"

def normalize_text(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s).strip()
    # collapse whitespace
    s = re.sub(r"\s+", " ", s)
    # remove trailing descriptors after colon (e.g., "Hogeschool Utrecht: Utrecht" -> "Hogeschool Utrecht")
    s = re.sub(r":.*$", "", s)
    return s

# Regex-based consolidation of common variants
NORMALIZATION_RULES = [
    (r"^NUOVO Scholen.*", "NUOVO Scholen"),
    (r"^Gooise Scholenfederatie.*", "Gooise Scholenfederatie"),
    (r"^Meerscholen.*", "Meerscholen"),
    (r"^Cals College.*", "Cals College"),
    (r"^Vallei College.*", "Vallei College"),
    (r"^College de Heemlanden.*", "College de Heemlanden"),
    (r"^Christelijk Lyceum Zeist.*", "Christelijk Lyceum Zeist"),
    (r"^Koningin Wilhelmina College.*", "Koningin Wilhelmina College"),
    (r"^Johannes Fontanus College.*", "Johannes Fontanus College"),
    (r"^Grafisch Lyceum Utrecht.*", "Grafisch Lyceum Utrecht"),
    (r"^Grafisch Lyceum Rotterdam.*", "Grafisch Lyceum Rotterdam"),
    (r"^Regionaal Opleidingencentrum van Twente$", "ROC van Twente"),
    (r"^Regionaal Opleidingen Centrum Rijn IJssel$", "ROC Rijn IJssel"),
    (r"^Regionaal Opleidingen Centrum Nova College$", "Nova College"),
    (r"^Regionaal Opleidingen Centrum Da Vinci College$", "Da Vinci College"),
    (r"^Regionaal Opleidingen Centrum ROC A12$", "ROC A12"),
    (r"^Regionaal Opleidingen Centrum Zadkine$", "Zadkine"),
    (r"^(Stg )?ROC Summa College$", "Summa College"),
    (r"^Stichting Landstede$", "Landstede"),
]

def apply_rules(s: str) -> str:
    if pd.isna(s):
        return s
    for pat, repl in NORMALIZATION_RULES:
        if re.match(pat, s):
            return repl
    return s

# --- Apply normalization and consolidation ---
data[col] = data[col].apply(normalize_text).apply(apply_rules)

# --- Add underscores between words ---
data[col] = data[col].str.replace(r"\s+", "_", regex=True)

# Quick audit
print("Unique categories after normalization:", data[col].nunique(dropna=False))
print("\nSample categories:")
print(data[col].value_counts(dropna=False))


In [ ]:
# Ultra-strict cleaner for Prior_Edu_School_Name_v1
# - Protect ROC, lowercase for matching, then restore ROC
# - Remove punctuation -> space
# - Remove edu abbreviations + generic words with a single regex (word-boundary)
# - Drop single-letter tokens and 2–4 letter alphabetic tokens (except 'roc')
# - Collapse spaces, replace with underscores, tidy underscores
# - If empty -> 'unknown' (to match your buckets)


col = "Prior_Edu_School_Name_v1"

# Words to remove (case-insensitive), matched as whole words with \b
EDU_ABBREVS = [
    "havo","mavo","vwo","vmbo","vbo","lwoo","pro","hbo","mbo","vavo","gym","ath"
]
GENERIC_WORDS = [
    "college","lyceum","school","scholengemeenschap",
    "universiteit","hogeschool","academie","onderwijs","opleiding","instituut"
]
DUTCH_STOP = [
    "de","het","een","en","of","van","der","den","op","aan","bij","met","naar",
    "voor","achter","onder","boven","tegen","tussen","door","over","tot","om",
    "in","uit","te","ten","ter","toe","af","per","als",
    # frequent abbreviations/labels that become tokens after punctuation strip
    "rk","pc","sgm","loc","locatie","bijz","openb","vest","ksg","bv","eo"
]

WORDLIST = EDU_ABBREVS + GENERIC_WORDS + DUTCH_STOP
WORDLIST_PAT = r"\b(" + "|".join(map(re.escape, WORDLIST)) + r")\b"

PUNCT = re.compile(r"[,\./\-()'’`\"]+")
WORDLIST_RE = re.compile(WORDLIST_PAT, flags=re.IGNORECASE)

def ultra_clean(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s).strip()

    # protect ROC in any case to keep it later
    s = re.sub(r"\broc\b", "__PROTECT_ROC__", s, flags=re.IGNORECASE)

    # remove trailing descriptor after colon (common pattern)
    s = re.sub(r":.*$", "", s)

    # punctuation -> space
    s = PUNCT.sub(" ", s)

    # lowercase for uniform matching
    s = s.lower()

    # drop whole-word items (edu abbrevs, generic words, stopwords)
    s = WORDLIST_RE.sub(" ", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove single-letter tokens and 2–4 letter alphabetic tokens except 'roc'
    tokens = s.split()
    filtered = []
    for t in tokens:
        if t == "__protect_roc__":  # handle if case changed (s is lower now)
            filtered.append("__PROTECT_ROC__")
            continue
        if len(t) == 1:
            continue
        if t.isalpha() and 2 <= len(t) <= 4 and t != "roc":
            continue
        filtered.append(t)

    # restore ROC
    filtered = ["ROC" if tok == "__PROTECT_ROC__" else tok for tok in filtered]

    if not filtered:
        return "unknown"

    # join -> underscores; tidy
    out = "_".join(filtered)
    out = re.sub(r"_+", "_", out).strip("_")
    return out

# apply
data[col] = data[col].apply(ultra_clean)

# audit
print("Unique categories after ultra cleaning:", data[col].nunique(dropna=False))
print("\nTop categories:")
print(data[col].value_counts(dropna=False))
